## Imports

In [ ]:
import pandas as pd
import numpy as np
import time
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler

dtype = torch.float
device = "cpu"
use_compile = False

if torch.cuda.is_available():
    device = "cuda"
    use_compile = (torch.cuda.get_device_capability() >= (7, 0))

torch.set_default_device(device)
torch.get_default_device()



# Test CUDA compatibility
if not use_compile:
    print("Warning: torch.compile is not supported on this device.")

## Data Load

### Data augmentation functions (not applied)

In [7]:
def augmentation_bernoulli(seq, prob=0.005):
    idx = torch.bernoulli(prob * torch.ones(len(seq))).nonzero().squeeze(dim=1)
    s = list(seq)

    for i in idx.tolist():
        s[i] = "N"

    return "".join(s)

def sequences_augmentation(data, level, cat, n):
    to_copy = data.loc[data[level] == cat]

    new_data = to_copy[0:1]
    new_data = new_data.drop(new_data.index[0])

    while new_data.shape[0] < n:
        qnt = ((n-(new_data.shape[0])) / to_copy.shape[0]).__ceil__()

        new_data = pd.concat(([to_copy]*qnt)+[new_data])
        new_data["truncated_sequence"] = new_data["truncated_sequence"].apply(augmentation_bernoulli, prob=0.002)
        new_data = new_data.drop_duplicates(subset=["truncated_sequence"])
    
    new_data = new_data[:n-to_copy.shape[0]]
    return new_data

def data_augmentation(data, level, lower, upper):
    class_count = data.groupby(level)[level].count().reset_index(name="count")
    
    cats = class_count.loc[(class_count["count"] < upper) & (class_count["count"] >= lower)][level].to_list()

    clones = sequences_augmentation(data, level, cats[0], upper)
    for cat in cats[1:]:
        clones = pd.concat([clones, sequences_augmentation(data, level, cat, upper)])

    return pd.concat([data, clones])


### Sequence encoder

In [8]:
base_map = {
    "A":[1.0, 0.0, 0.0, 0.0],
    "T":[0.0, 1.0, 0.0, 0.0],
    "G":[0.0, 0.0, 1.0, 0.0],
    "C":[0.0, 0.0, 0.0, 1.0],

    'W':[0.5, 0.5, 0.0, 0.0],
    'S':[0.0, 0.0, 0.5, 0.5],
    'M':[0.5, 0.0, 0.0, 0.5],
    'K':[0.0, 0.5, 0.5, 0.0],
    'R':[0.5, 0.0, 0.5, 0.0],
    'Y':[0.0, 0.5, 0.0, 0.5],
    
    'B':[0.0, 0.3, 0.3, 0.3],
    'D':[0.3, 0.3, 0.3, 0.0],
    'H':[0.3, 0.3, 0.0, 0.3],
    'V':[0.3, 0.0, 0.3, 0.3],

    'N':[0.25, 0.25, 0.25, 0.25],
}

def encode_sequence(sequence):
    encoded_seq = []

    for base in sequence:
        encoded_seq.append(base_map[base])
    
    return torch.tensor(encoded_seq)

### PyTorch dataset object to load Sequences and Classification Data

In [9]:
class SequenceDataset(Dataset):
    def __init__(self, train, test, level, augmentation=False):

        self.classes = pd.concat([train[level], test[level]]).unique().tolist()
        self.classes.sort()
        self.level = level

        if augmentation:
            train = data_augmentation(train, level, 10, 500)
        
        self.labels = train[level]
        self.encoded_labels = SequenceDataset.__encoded_labels__(self.classes, self.labels)
        self.sequences = SequenceDataset.__sequences__(train)

        self.test = SequenceDatasetTest(
            labels = test[level],
            classes = self.classes,
            encoded_labels = SequenceDataset.__encoded_labels__(self.classes, test[level]),
            sequences = SequenceDataset.__sequences__(test)
            )

    def __encoded_labels__(classes, labels):
        return torch.nn.functional.one_hot(torch.tensor([classes.index(l) for l in labels]), len(classes)).type(torch.cuda.FloatTensor)
    
    def __sequences__(ds):
        sequences = []
        for _, row in ds.iterrows():
            sequences.append(encode_sequence(row["truncated_sequence"]))        
        return torch.stack(sequences, dim=0)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return   self.sequences[idx], self.encoded_labels[idx]
    
    def __getitems__(self, ids):
        idx = torch.tensor(ids, device=torch.device('cuda:0'))
        return   list(zip(torch.index_select(self.sequences, 0, idx), torch.index_select(self.encoded_labels, 0, idx)))
    
    def get_test(self):
        return self.test

class SequenceDatasetTest(SequenceDataset):    
    def __init__(self, labels, classes, encoded_labels, sequences):
        self.labels = labels
        self.classes = classes
        self.encoded_labels = encoded_labels
        self.sequences = sequences

### Generate PyTorch DataLoader objects

In [10]:
def loaders_generator(ds_train, ds_test, bs = 128):
    train_loader = DataLoader(ds_train, batch_size=bs, shuffle=True, generator=torch.Generator(device='cuda'))
    test_loader = DataLoader(ds_test, batch_size=bs, shuffle=True, generator=torch.Generator(device='cuda'))

    return train_loader, test_loader

## Models

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=4):
        super(ResidualBlock, self).__init__()
        
        # Padding to maintain input size
        self.padding = nn.CircularPad1d((1,2))
        
        # Convolutional layers
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size)
        self.bn1 = nn.BatchNorm1d(out_channels)
        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Conv1d(in_channels, out_channels, kernel_size=1)
        
        # Activation
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        # Store the input for the residual connection
        residual = x
        
        # Main path
        out = self.padding(x)
        out = self.conv1(out)
        out = self.bn1(out)
        
        # Shortcut connection
        residual = self.shortcut(residual)
        
        # Add residual connection
        out += residual
        out = self.relu(out)
        
        return out

In [ ]:
class ResidualBlockGELU(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=4):
        super(ResidualBlockGELU, self).__init__()
        
        # Padding to maintain input size
        self.padding = nn.CircularPad1d((1,2))
        
        # Convolutional layers
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size)
        self.bn1 = nn.BatchNorm1d(out_channels)
        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Conv1d(in_channels, out_channels, kernel_size=1)
        
        # Activation
        self.act = nn.GELU()
    
    def forward(self, x):
        # Store the input for the residual connection
        residual = x
        
        # Main path
        out = self.padding(x)
        out = self.conv1(out)
        out = self.bn1(out)
        
        # Shortcut connection
        residual = self.shortcut(residual)
        
        # Add residual connection
        out += residual
        out = self.act(out)
        
        return out

In [16]:
class SimplestCNNClassifier_6layers_Residual(nn.Module):
    def __init__(self, nClasses):
        super(SimplestCNNClassifier_6layers_Residual, self).__init__()
        
        # Residual blocks with adaptive pooling
        self.residual_block1 = ResidualBlock(4, 16)
        self.adAvgPool1 = nn.AdaptiveAvgPool1d(450)
        
        self.residual_block2 = ResidualBlock(16, 32)
        self.adAvgPool2 = nn.AdaptiveAvgPool1d(225)
        
        self.residual_block3 = ResidualBlock(32, 64)
        self.adAvgPool3 = nn.AdaptiveAvgPool1d(112)
        
        self.residual_block4 = ResidualBlock(64, 128)
        self.adAvgPool4 = nn.AdaptiveAvgPool1d(56)
        
        # Two additional residual blocks
        self.residual_block5 = ResidualBlock(128, 256)
        self.adAvgPool5 = nn.AdaptiveAvgPool1d(28)
        
        self.residual_block6 = ResidualBlock(256, 512)
        self.adAvgPool6 = nn.AdaptiveAvgPool1d(14)
        
        # Activation and fully connected layers
        self.act = nn.ReLU()
        
        # Calculate the input size for linear layers
        # Note: You might need to adjust this based on your specific input dimensions
        self.linear1 = nn.Linear(7168, 7168)
        self.linear2 = nn.Linear(7168, nClasses)
    
    def forward(self, x):
        # Move channel dimension
        x = torch.movedim(x, -1, -2)
        
        # First residual block
        x = self.residual_block1(x)
        x = self.adAvgPool1(x)
        
        # Second residual block
        x = self.residual_block2(x)
        x = self.adAvgPool2(x)
        
        # Third residual block
        x = self.residual_block3(x)
        x = self.adAvgPool3(x)
        
        # Fourth residual block
        x = self.residual_block4(x)
        x = self.adAvgPool4(x)
        
        # Fifth residual block
        x = self.residual_block5(x)
        x = self.adAvgPool5(x)
        
        # Sixth residual block
        x = self.residual_block6(x)
        x = self.adAvgPool6(x)
        
        # Flatten and fully connected layers
        x = torch.flatten(x, 1)
        x = self.linear1(x)
        x = self.act(x)
        x = self.linear2(x)
        
        return x

In [17]:
class SimplestCNNClassifier_8layers_Residual(nn.Module):
    def __init__(self, nClasses):
        super(SimplestCNNClassifier_8layers_Residual, self).__init__()
        
        # Residual blocks with adaptive pooling
        self.residual_block1 = ResidualBlock(4, 16)
        self.adAvgPool1 = nn.AdaptiveAvgPool1d(450)
        
        self.residual_block2 = ResidualBlock(16, 32)
        self.adAvgPool2 = nn.AdaptiveAvgPool1d(225)
        
        self.residual_block3 = ResidualBlock(32, 64)
        self.adAvgPool3 = nn.AdaptiveAvgPool1d(112)
        
        self.residual_block4 = ResidualBlock(64, 128)
        self.adAvgPool4 = nn.AdaptiveAvgPool1d(56)
        
        self.residual_block5 = ResidualBlock(128, 256)
        self.adAvgPool5 = nn.AdaptiveAvgPool1d(28)
        
        self.residual_block6 = ResidualBlock(256, 512)
        self.adAvgPool6 = nn.AdaptiveAvgPool1d(14)
        
        # Two additional residual blocks
        self.residual_block7 = ResidualBlock(512, 1024)
        self.adAvgPool7 = nn.AdaptiveAvgPool1d(7)
        
        self.residual_block8 = ResidualBlock(1024, 2048)
        self.adAvgPool8 = nn.AdaptiveAvgPool1d(3)
        
        # Activation and fully connected layers
        self.act = nn.ReLU()
        
        # Calculate the input size for linear layers
        # Note: You might need to adjust this based on your specific input dimensions
        self.linear1 = nn.Linear(6144, 6144)
        self.linear2 = nn.Linear(6144, nClasses)
    
    def forward(self, x):
        # Move channel dimension
        x = torch.movedim(x, -1, -2)
        
        # First residual block
        x = self.residual_block1(x)
        x = self.adAvgPool1(x)
        
        # Second residual block
        x = self.residual_block2(x)
        x = self.adAvgPool2(x)
        
        # Third residual block
        x = self.residual_block3(x)
        x = self.adAvgPool3(x)
        
        # Fourth residual block
        x = self.residual_block4(x)
        x = self.adAvgPool4(x)
        
        # Fifth residual block
        x = self.residual_block5(x)
        x = self.adAvgPool5(x)
        
        # Sixth residual block
        x = self.residual_block6(x)
        x = self.adAvgPool6(x)
        
        # Seventh residual block
        x = self.residual_block7(x)
        x = self.adAvgPool7(x)
        
        # Eighth residual block
        x = self.residual_block8(x)
        x = self.adAvgPool8(x)
        
        # Flatten and fully connected layers
        x = torch.flatten(x, 1)
        x = self.linear1(x)
        x = self.act(x)
        x = self.linear2(x)
        
        return x

## Test Params

In [31]:
levels = [
    "class", 
    "order", 
    "family", 
    "genus",
    "species",
]

In [32]:
batch_sizes = [
    # 64,
    # 128,
    # 256,
    # 512,
    # 2048,
    # 10000,
    "dynamic"
]

In [33]:
epochs = [
    # 1,
    # 20,
    # 50,
    # 100,
    # 150,
    # 200,
    # 300,
    # 500,
    # 600,
    700,
    # 1000,
]

In [34]:
models_list = [
    SimplestCNNClassifier_8layers_Residual,
]

In [35]:
loss_functions = {
    "CrossEntropyLoss":{
        "function":nn.CrossEntropyLoss,
        "params":{},
        "function_params":{}
    },
}

In [36]:
learning_rates = [
    # 5e-2,
    # 1e-2,
    5e-3,
    # 1e-3,
    # 5e-4,
    # 1e-4,
]

In [37]:
optimizers = [
    {
        "optim":torch.optim.AdamW,
        "params":{
            "weight_decay":1.0,
            "amsgrad":True
        }
    },
]

In [38]:
hiperparams = {
    "batch_size": batch_sizes,
    "epochs": epochs,
    "model": models_list,
    "loss_function": loss_functions,
    "learning_rate": learning_rates,
    "optimizer": optimizers    
}

In [39]:
hiperparams

{'batch_size': ['dynamic'],
 'epochs': [700],
 'model': [__main__.SimplestCNNClassifier_8layers_Residual],
 'loss_function': {'CrossEntropyLoss': {'function': torch.nn.modules.loss.CrossEntropyLoss,
   'params': {},
   'function_params': {}}},
 'learning_rate': [0.005],
 'optimizer': [{'optim': torch.optim.adamw.AdamW,
   'params': {'weight_decay': 1.0, 'amsgrad': True}}]}

## Batch Execution

### Train and Test function

In [40]:
def Train_Test(model, loss_fn, optimizer, epochs, learning_rate, batch_size, train_data,test_data,id=""):
    
    print("Model: \t\t\t"+(model._get_name() if not model._get_name() == "OptimizedModule" else model.__dict__["_modules"]["_orig_mod"].__class__.__name__))
    print("  Loss Func.: \t\t"+loss_fn._get_name())
    print("  Optimizer: \t\t"+type(optimizer).__name__)
    print("  Epochs: \t\t"+str(epochs))
    print("  Learning Rate: \t"+str(learning_rate))

    print("\nModel Arch: ")
    print(str(model))
    print("\n\n\n")


    epochs_results = []
    current = {
        "model":(model._get_name() if not model._get_name() == "OptimizedModule" else model.__dict__["_modules"]["_orig_mod"].__class__.__name__),
        "loss_function":loss_fn._get_name(),
        "epoch":None,
        "learning_rate":learning_rate,
        "batch_size":None,
        "train_size":None,
        "test_size":None,
        "optimizer":type(optimizer).__name__,
        "train_acc":None,
        "train_loss":None,
        "test_acc":None,
        "test_loss":None,
    }

    # Prepare batch sizes to use
    if batch_size == "dynamic":
        bss = [15000, 15000, 15000, 1000, 15000, 15000, 15000, 15000, 256, 15000, 15000, 15000, 15000, 128, 15000]
    else:
        bss = [batch_size]
    if len(bss) > epochs:
        bss = bss[0:epochs]
    print("Batch Sizes List: "+str(bss))
    batch_lim = int(epochs/len(bss))
    
    
    t_start = time.time()
    best = {
        "epoch":0,
        "train_acc":0,
        "train_loss":10000000,
        "test_acc":0,
        "test_loss":10000000,
    }
    
    train_loader = None
    test_loader = None
    
    
    scaler = GradScaler(device=device)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=10,  # First restart
        T_mult=2,  # Period multiplier
        eta_min=1e-10,  # Minimum learning rate
    )

    # Epochs
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}\n-------------------------------")

        # Create DataLoaders with current batch size
        if epoch%batch_lim == 0 and len(bss) > 0:
            if train_loader:
                del train_loader
            if test_loader:
                del test_loader

            batch_size = bss.pop(0)
            train_loader, test_loader = loaders_generator(train_data, test_data, batch_size)

        print("Batch Size: "+str(batch_size))
        
        
        # Train Phase

        model.train()
        train_loss = 0
        train_acc = 0
        
        # Run train over the batches
        optimizer.zero_grad()
        for batch, (X, y) in enumerate(train_loader):
            with torch.autocast(device_type=device, dtype=torch.float16):
                # Compute prediction and loss
                pred = model(X)
                loss = loss_fn(pred, y)
                
            # Backpropagation
            scaler.scale(loss).backward()

            # Gradien clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
            
            scaler.step(optimizer)
            scaler.update()
            
            # Update the learning rate
            scheduler.step(epoch + batch / len(train_loader))

            optimizer.zero_grad()

            # Update results
            train_loss += loss.item()
            train_acc += (pred.argmax(1) == y.argmax(1)).type(torch.float).sum().item()

        # Train results
        train_loss /= len(train_loader)
        train_acc /= len(train_loader.dataset)
        print("Last Learning Rate: "+str(scheduler.get_last_lr()[0]))
        print(f"Train Error: \n Accuracy: {(100*train_acc):>0.1f}%, Avg loss: {train_loss:>8f} \n")


        # Test Phase
        model.eval()
        test_loss = 0
        test_acc = 0

        with torch.no_grad():
            for X, y in test_loader:
                pred = model(X)
                test_loss += loss_fn(pred, y).item()
                test_acc += (pred.argmax(1) == y.argmax(1)).type(torch.float).sum().item()

        # Test results
        test_loss /= len(test_loader)
        test_acc /= len(test_loader.dataset)
        print(f"Test Error: \n Accuracy: {(100*test_acc):>0.1f}%, Avg loss: {test_loss:>8f} \n")


        # Update Results
        if best["test_acc"] < test_acc or (best["test_acc"] == test_acc and best["train_acc"] < train_acc):
            best["epoch"] = epoch+1
            best["test_acc"] = test_acc
            best["test_loss"] = test_loss
            best["train_acc"] = train_acc
            best["train_loss"] = train_loss

            # If accuracy over 50%, export the current best treined model
            if test_acc > 0.5:
                torch.save(model.state_dict(), "./models/"+train_data.level+"/"+str(id)+"_"+current["model"]+".pth")
                                
                    
        current["epoch"] = epoch+1
        current["batch_size"] = batch_size
        current["learning_rate"] = scheduler.get_last_lr()[0]
        current["train_size"] = train_loader.dataset.__len__()
        current["test_size"] = test_loader.dataset.__len__()
        current["train_acc"] = train_acc
        current["train_loss"] = train_loss
        current["test_acc"] = test_acc
        current["test_loss"] = test_loss

        epochs_results.append(current.copy())

    # Save Train/Test iteration information
    pd.DataFrame(epochs_results).to_csv("./results/epochs/"+str(id)+"__"+current["model"]+"_train_test.csv")
    
    print("\n\n")
    print(f"Best Epoch:{best['epoch']} \n\tAccuracy: {(100*best['test_acc']):>0.1f}%, Avg loss: {best['test_loss']:>8f} \n")
    print("Train and Test execution time: "+str(format(time.time()-t_start, '.4f'))+"s")
    print("Done!")

    return best

## Experiment Iterations

In [41]:
!mkdir results
!mkdir ./results/summarized
!mkdir ./results/epochs
!mkdir models
!mkdir ./models/class
!mkdir ./models/order
!mkdir ./models/family
!mkdir ./models/genus
!mkdir ./models/species

mkdir: cannot create directory ‘results’: File exists
mkdir: cannot create directory ‘./results/summarized’: File exists
mkdir: cannot create directory ‘./results/epochs’: File exists
mkdir: cannot create directory ‘models’: File exists
mkdir: cannot create directory ‘./models/class’: File exists
mkdir: cannot create directory ‘./models/order’: File exists
mkdir: cannot create directory ‘./models/family’: File exists
mkdir: cannot create directory ‘./models/genus’: File exists
mkdir: cannot create directory ‘./models/species’: File exists


In [ ]:
import gc

# Global references
_model_ = None
_lossfunction_ = None
_optimizer_ = None

# Function to clean cache
def clear():
    global _model_, _lossfunction_, _optimizer_
    
    torch.cuda.empty_cache()
    torch.compiler.reset()
    torch._dynamo.reset()

    if _model_:
        del _model_
        _model_ = None
    if _lossfunction_:
        del _lossfunction_
        _lossfunction_ = None
    if _optimizer_:
        del _optimizer_
        _optimizer_ = None
    
    torch.cuda.empty_cache()
    gc.collect()

results = []
current = {}

id = 0
time_id = str(int(time.time()))
print("Time ID: "+str(time_id))

splitters = [
    f"prop_{str(prop)}/min_{kmin}/{splitter}_{seed}"
    for kmin in [10,5]
    for (splitter, props) in 
    [
        ("StratifiedSplit2", ['0-2', '0-1']), 
        ("RandomSplit", ['0-2', '0-1', '0-05'])
    ]
    for prop in props
    for seed in [0, 14, 56, 92, 84, 101, 105, 227] 
]

apply_augmentation = [
    False,
    # True,
]

for mat_mul in [False]:
    for augmentation in apply_augmentation:
        for level in levels:
            for splitter in splitters:
                clear()

                path = "../new_data/"+splitter+"/"+level
                if (not os.path.exists(path+"/train_dataset.csv")) or (not os.path.exists(path+"/test_dataset.csv")):
                    print("\nError:")
                    print(f'\tNo train/test data files found at: {path}')
                    continue
                    
                # Load train and test datasets
                train_data = pd.read_csv(path+"/train_dataset.csv")
                test_data  = pd.read_csv(path+"/test_dataset.csv")
                print(level)
                print(train_data.shape)
                print(test_data.shape)

                dataset = SequenceDataset(
                    train=train_data, 
                    test=test_data, 
                    level=level, 
                    augmentation=augmentation)
                
                print(dataset.__len__())
                print(dataset.test.__len__())


                for batch_size in hiperparams["batch_size"]:
                    for epochs in hiperparams["epochs"]:
                        for model in hiperparams["model"]:
                            for loss_function_name, loss_function in hiperparams["loss_function"].items():
                                for learning_rate in hiperparams["learning_rate"]:
                                    for optimizer in hiperparams["optimizer"]:
                                        clear()
                                        
                                        optim = optimizer["optim"]
                                        optim_params = optimizer["params"] if "params" in optimizer.keys() else {}

                                        current = {
                                                "id": id,
                                                "start_time":time.time(),
                                                "end_time": None,
                                                "level": level,
                                                "splitter": splitter,
                                                "augmentation": augmentation,
                                                "batch_size": batch_size,
                                                "epochs": epochs,
                                                "model": model.__name__,
                                                "loss_function": loss_function_name+" ("+str(loss_function["function"])+")",
                                                "learning_rate": learning_rate,
                                                "optimizer": optim.__name__+" (params: "+str(optim_params)+")",
                                                "mat_mul": mat_mul,
                                                "reserved_memory": None,
                                                "error": None
                                            }


                                        try:
                                            # Change precision to improve model performance 
                                            # if mat_mul:
                                            #     torch.set_float32_matmul_precision('high')
                                            # else:
                                            #     torch.set_float32_matmul_precision('highest')
                                            
                                            # Initialize a compiled model, loss function, and optimizer
                                            _model_ = model(dataset.encoded_labels.shape[1])
                                            _model_ = (torch.compile(_model_) if use_compile else _model_)
                                            _lossfunction_ = loss_function["function"](**{func:params[0](*params[1:]) for func,params in loss_function["function_params"].items()})
                                            _optimizer_ = optim(_model_.parameters(), lr=learning_rate, **optim_params)


                                            # Runt Train-Test
                                            result = Train_Test(
                                                model=_model_,
                                                loss_fn=_lossfunction_,
                                                optimizer=_optimizer_,
                                                epochs=epochs,
                                                learning_rate=learning_rate,
                                                batch_size=batch_size,
                                                train_data=dataset,
                                                test_data=dataset.get_test(),
                                                id=time_id+"_"+str(id),
                                                )
                                                
                                            current["end_time"] = time.time()
                                            current["best_epoch"] = result["epoch"]
                                            current["train_acc_best_epoch"] = result["train_acc"]
                                            current["train_loss_best_epoch"] = result["train_loss"]
                                            current["test_acc_best_epoch"] = result["test_acc"]
                                            current["test_loss_best_epoch"] = result["test_loss"]

                                            current["reserved_memory"] = torch.cuda.memory_reserved() / 1024 / 1024  # Convert to MB

                                            clear()                                
                                            
                                        except Exception as e:
                                            print(e)
                                            current["error"] = str(e)
                                        
                                        # Save the results
                                        results.append(current)
                                        pd.DataFrame(results).to_csv("./results/summarized/"+str(time_id)+"_models_train_test_"+str(len(results))+".csv")
                                        
                                        id = id+1

clear()

Time ID: 1773593582
Error:
	No train/test data files found at: /kaggle/input/datasets/gsfrainer/new-data/new_data/prop_0-2/min_10/RandomSplit_0/class1
Error:
	No train/test data files found at: /kaggle/input/datasets/gsfrainer/new-data/new_data/prop_0-2/min_10/RandomSplit_0/order1
family
(66372, 11)
(16593, 11)
66372
16593
Model: 			SimplestCNNClassifier_8layers_Residual
  Loss Func.: 		CrossEntropyLoss
  Optimizer: 		AdamW
  Epochs: 		700
  Learning Rate: 	0.005

Model Arch: 
SimplestCNNClassifier_8layers_Residual(
  (residual_block1): ResidualBlock(
    (padding): CircularPad1d((1, 2))
    (conv1): Conv1d(4, 16, kernel_size=(4,), stride=(1,))
    (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Conv1d(4, 16, kernel_size=(1,), stride=(1,))
    (relu): ReLU(inplace=True)
  )
  (adAvgPool1): AdaptiveAvgPool1d(output_size=450)
  (residual_block2): ResidualBlock(
    (padding): CircularPad1d((1, 2))
    (conv1): Conv1d(16, 32, kernel_s